# AcademicRag — rebuild index, retrieval ablation, local-LLM generation (Colab T4)

Runtime → Change runtime type → **T4 GPU**. Expects `scholarrag_data.zip` (MinerU-parsed papers + train/eval sets) in the root of your Google Drive.

In [ ]:
!nvidia-smi -L
!rm -rf /content/AcademicRag && git clone -q https://github.com/Ayaan20/AcademicRag /content/AcademicRag
%cd /content/AcademicRag
!pip -q install FlagEmbedding faiss-cpu python-dotenv 2>&1 | tail -2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/scholarrag_data.zip -d /content/AcademicRag && ls parsed_pdf | wc -l

## 1. Chunk 100 papers + embed with BGE-M3 (dense + learned sparse) + build FAISS

In [ ]:
!python rebuild_index.py 2>&1 | grep -v 'it/s\]' | tail -40

## 2. Retrieval ablation: BM25 vs dense vs hybrid (RRF) vs hybrid + cross-encoder (204 held-out questions)

In [ ]:
!python run_ablation.py --dataset evaluation_set_100papers.json 2>&1 | grep -E '^\||Ablation|ERROR|Error|Traceback' | tail -30
!cat eval_results/ablation_retrieval.md

## 3. Local generation with an open-weight model served by Ollama

In [ ]:
!apt-get -qq install -y zstd > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1; which ollama
import subprocess, time
subprocess.Popen(['ollama','serve'], stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen2.5:7b-instruct 2>&1 | tail -1

In [ ]:
import os
os.environ['LLM_BASE_URL']='http://localhost:11434/v1'
os.environ['LLM_MODEL']='qwen2.5:7b-instruct'
!python batch_evaluate.py --dataset evaluation_set_100papers.json --max 30 --output eval_results/generation_local 2>&1 | tail -40

## 4. Save results (and the built index) to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/academicrag_results && cp -r eval_results /content/drive/MyDrive/academicrag_results/ && zip -q -r /content/drive/MyDrive/academicrag_results/index_output.zip src/embedding/output && ls -la /content/drive/MyDrive/academicrag_results